# BERT: год сырых посадок, прогноз на неделю

Посадки считаются из сырого `train.csv` и `test.csv`: строка с `validation_result == 1`. В BERT не идут отдельные валидации и не идёт текстовое описание часа.

Один пример — один маршрут и до 365 последних дней. День — это 24 числа посадок. Цель — следующие 7 дней (168 часов). Данных до 1 января 2025 нет, поэтому к 1 сентября в контексте 243 дня, к ноябрю — 304.

Обучение: недели с концом не позже 31 августа. Скор сентября–октября — это последовательные недели вперёд по настоящей истории, до дообучения на этих метках. `submission.csv` на ноябрь–декабрь крутит ту же неделю рекурсивно.


In [ ]:
from datetime import date, timedelta
from pathlib import Path
import pickle

import pandas as pd

ROUTES = {1, 5, 7, 11, 12, 17, 25, 26, 28, 50}
ROUTE_LIST = [1, 5, 7, 11, 12, 17, 25, 26, 28, 50]
KEEP_FROM = "2025-01-01"
KEEP_TO = "2025-10-31"
TRAIN_START = date(2025, 1, 1)
TRAIN_END = date(2025, 8, 31)
TEST_START = date(2025, 9, 1)
TEST_END = date(2025, 10, 31)

direct = Path("/kaggle/input/datasets/kirillvorotnikov2007/mstrans")
if (direct / "train.csv").exists() and (direct / "test.csv").exists():
    data_root = direct
else:
    data_root = next(Path("/kaggle/input").rglob("train.csv")).parent
train_path = data_root / "train.csv"
test_path = data_root / "test.csv"
print("train", train_path)
print("test", test_path)


def separator(path):
    preview = pd.read_csv(path, sep=";", encoding="utf-8-sig", nrows=1)
    sep = ";"
    if "tran_date_time" not in preview.columns:
        preview = pd.read_csv(path, sep=",", encoding="utf-8-sig", nrows=1)
        sep = ","
    print(path.name, "sep", sep, list(preview.columns))
    return sep


def daterange(start, end):
    day = start
    while day <= end:
        yield day
        day += timedelta(days=1)


def accumulate(path, sep):
    counts = {}
    rows = kept = 0
    reader = pd.read_csv(
        path,
        sep=sep,
        encoding="utf-8-sig",
        usecols=["tran_date_time", "validation_result", "ngpt_route"],
        chunksize=1_000_000,
        dtype="string",
        on_bad_lines="skip",
    )
    for chunk_no, chunk in enumerate(reader, start=1):
        rows += len(chunk)
        valid = chunk["validation_result"].fillna("").str.replace(".0", "", regex=False).str.strip().eq("1")
        part = chunk.loc[valid, ["tran_date_time", "ngpt_route"]]
        if not part.empty:
            route_num = pd.to_numeric(part["ngpt_route"].str.extract(r"(\d+)", expand=False), errors="coerce")
            text = part["tran_date_time"].fillna("").str.strip().str.replace("T", " ", regex=False)
            iso = text.str.slice(0, 10).str.match(r"\d{4}-\d{2}-\d{2}", na=False)
            dmy = text.str.slice(0, 10).str.match(r"\d{2}\.\d{2}\.\d{4}", na=False)
            times = pd.Series(pd.NaT, index=part.index)
            if bool(iso.any()):
                times.loc[iso] = pd.to_datetime(text.loc[iso].str.slice(0, 13), format="%Y-%m-%d %H", errors="coerce")
            if bool(dmy.any()):
                times.loc[dmy] = pd.to_datetime(text.loc[dmy].str.slice(0, 13), format="%d.%m.%Y %H", errors="coerce")
            frame = pd.DataFrame({"route": route_num.to_numpy(), "time": times.to_numpy()})
            frame = frame.dropna()
            frame = frame.loc[frame["route"].isin(ROUTES)]
            frame["day"] = frame["time"].dt.strftime("%Y-%m-%d")
            frame["hour"] = frame["time"].dt.hour.astype(int)
            frame = frame.loc[(frame["day"] >= KEEP_FROM) & (frame["day"] <= KEEP_TO)]
            grouped = frame.groupby(["route", "day", "hour"], sort=False).size()
            for (route, day, hour), value in grouped.items():
                key = (int(route), str(day), int(hour))
                counts[key] = counts.get(key, 0) + int(value)
                kept += int(value)
        print(f"{path.name} chunk {chunk_no} rows {rows} boardings {kept}", flush=True)
    return counts, rows, kept


def fill_grid(counts, start, end):
    grid = {}
    outside = 0
    for (route, day, hour), value in counts.items():
        current = date.fromisoformat(day)
        if start <= current <= end:
            grid[(route, current, hour)] = float(value)
        else:
            outside += int(value)
    for current in daterange(start, end):
        for hour in range(24):
            for route in ROUTE_LIST:
                grid.setdefault((route, current, hour), 0.0)
    return grid, outside


train_counts, train_rows, train_kept = accumulate(train_path, separator(train_path))
test_counts, test_rows, test_kept = accumulate(test_path, separator(test_path))
history, train_outside = fill_grid(train_counts, TRAIN_START, TRAIN_END)
test_actual, test_outside = fill_grid(test_counts, TEST_START, TEST_END)
print(
    "train rows", train_rows, "boardings", train_kept, "grid", sum(history.values()), "outside", train_outside,
    flush=True,
)
print(
    "test rows", test_rows, "boardings", test_kept, "grid", sum(test_actual.values()), "outside", test_outside,
    flush=True,
)

out = Path("/kaggle/working")
with open(out / "history.pkl", "wb") as handle:
    pickle.dump(history, handle, protocol=4)
with open(out / "test_actual.pkl", "wb") as handle:
    pickle.dump(test_actual, handle, protocol=4)

labels = {}
for path in Path("/kaggle/input").rglob("labels_day_*.csv"):
    table = pd.read_csv(path, sep=";", encoding="utf-8-sig")
    for row in table.itertuples(index=False):
        labels[(int(row.route), str(row.date)[:10], int(row.hour))] = float(row.boardings)
sparse = {}
for source in (train_counts, test_counts):
    for key, value in source.items():
        current = date.fromisoformat(key[1])
        if TRAIN_START <= current <= TEST_END:
            sparse[key] = sparse.get(key, 0) + value
if labels:
    keys = set(sparse) | set(labels)
    gap = sum(abs(sparse.get(key, 0) - labels.get(key, 0.0)) for key in keys)
    print("abs gap vs labels", round(gap, 1), "sparse hours", len(sparse), "label hours", len(labels), flush=True)


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "einops"])
subprocess.check_call(["nvidia-smi", "-L"])
Path("/kaggle/working/train_ddp.py").write_text(
"\"\"\"Год почасовых посадок из сырого CSV → следующая неделя.\n\nКонтекст — до 365 последних дней маршрута, каждый день это 24 числа посадок.\nЦель — 168 часов сразу после этого окна. Прогноз стартует с копии предыдущей\nнедели, BERT добавляет поправку.\n\"\"\"\n\nimport os\nimport pickle\nfrom datetime import date, timedelta\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.distributed as dist\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom torch.utils.data import DataLoader, Dataset, DistributedSampler\nfrom transformers import AutoConfig, AutoModelForMaskedLM\n\nOUT = Path(\"/kaggle/working\")\nROUTES = [1, 5, 7, 11, 12, 17, 25, 26, 28, 50]\nSTART = date(2025, 1, 1)\nTRAIN_END = date(2025, 8, 31)\nTEST_START = date(2025, 9, 1)\nTEST_END = date(2025, 10, 31)\nFORECAST_START = date(2025, 11, 1)\nFORECAST_END = date(2025, 12, 31)\nMAX_DAYS = 365\nMIN_DAYS = 28\nHORIZON = 7\nBATCH = 4\nEPOCHS_TRAIN = 5\nEPOCHS_ALL = 1\nENC_LR = 1e-5\nHEAD_LR = 1e-3\nWARMUP = 40\n\n\ndef daterange(start, end):\n    day = start\n    while day <= end:\n        yield day\n        day += timedelta(days=1)\n\n\ndef points_of(score):\n    if score <= 0.48:\n        return 0\n    if score <= 0.60:\n        return 1\n    if score <= 0.70:\n        return 2\n    if score <= 0.80:\n        return 3\n    if score <= 0.88:\n        return 4\n    return 5\n\n\ndef pin_alibi(encoder):\n    if \"alibi\" in encoder._buffers:\n        return\n    alibi = encoder.alibi.detach().clone()\n    del encoder.alibi\n    encoder.register_buffer(\"alibi\", alibi)\n\n\nclass WeekBert(nn.Module):\n    def __init__(self):\n        super().__init__()\n        config = AutoConfig.from_pretrained(\"mosaicml/mosaic-bert-base\", trust_remote_code=True)\n        mlm = AutoModelForMaskedLM.from_config(config, trust_remote_code=True)\n        self.bert = mlm.bert\n        hidden = config.hidden_size\n        self.day_proj = nn.Sequential(\n            nn.Linear(24, hidden),\n            nn.LayerNorm(hidden),\n        )\n        self.head = nn.Linear(hidden, HORIZON * 24)\n        self.mix = nn.Parameter(torch.tensor(-4.0))\n        nn.init.normal_(self.head.weight, std=1e-3)\n        nn.init.zeros_(self.head.bias)\n\n    def forward(self, days, mask):\n        embeds = self.day_proj(torch.log1p(days))\n        encoded = self.bert.encoder(embeds, mask, output_all_encoded_layers=False)\n        hidden = encoded[-1]\n        if hidden.shape[1] != days.shape[1]:\n            raise RuntimeError(f\"encoder вернул последовательность {tuple(hidden.shape)}\")\n        delta = self.head(hidden[:, -1].float())\n        previous = days[:, -HORIZON:, :].reshape(days.shape[0], -1).float()\n        correction = torch.sigmoid(self.mix) * delta\n        return (previous + correction).clamp(min=0), previous, correction\n\n\nclass WeekSet(Dataset):\n    def __init__(self, days, series, targets):\n        self.days = days\n        self.series = series\n        self.targets = targets\n        self.index = []\n        for route_i, _route in enumerate(ROUTES):\n            for target in targets:\n                self.index.append((route_i, target))\n\n    def __len__(self):\n        return len(self.index)\n\n    def __getitem__(self, item):\n        route_i, target = self.index[item]\n        ctx = self.series[route_i][max(0, target - MAX_DAYS):target]\n        padded = np.zeros((MAX_DAYS, 24), np.float32)\n        mask = np.zeros(MAX_DAYS, np.float32)\n        padded[-len(ctx):] = ctx\n        mask[-len(ctx):] = 1.0\n        future = self.series[route_i][target:target + HORIZON].reshape(-1)\n        return (\n            torch.from_numpy(padded),\n            torch.from_numpy(mask),\n            torch.from_numpy(future.copy()),\n            len(ctx),\n        )\n\n\ndef load_series():\n    with open(OUT / \"history.pkl\", \"rb\") as handle:\n        history = pickle.load(handle)\n    with open(OUT / \"test_actual.pkl\", \"rb\") as handle:\n        history.update(pickle.load(handle))\n    days = list(daterange(START, TEST_END))\n    position = {day: index for index, day in enumerate(days)}\n    series = np.zeros((len(ROUTES), len(days), 24), np.float32)\n    route_pos = {route: index for index, route in enumerate(ROUTES)}\n    for (route, day, hour), value in history.items():\n        row = route_pos.get(route)\n        col = position.get(day)\n        if row is None or col is None:\n            continue\n        series[row, col, hour] = value\n    return days, series\n\n\ndef target_starts(days, last_day):\n    starts = []\n    for index, day in enumerate(days):\n        if index < MIN_DAYS or index + HORIZON > len(days):\n            continue\n        if days[index + HORIZON - 1] <= last_day:\n            starts.append(index)\n    return starts\n\n\ndef make_loader(dataset, rank, world):\n    sampler = DistributedSampler(dataset, num_replicas=world, rank=rank, shuffle=True, drop_last=False)\n    loader = DataLoader(dataset, batch_size=BATCH, sampler=sampler, num_workers=0, pin_memory=True)\n    return loader, sampler\n\n\ndef fit(model, loader, sampler, optimizer, scaler, epochs, rank):\n    step = 0\n    for epoch in range(epochs):\n        sampler.set_epoch(epoch)\n        model.train()\n        total_abs = base_abs = total_y = 0.0\n        for days, mask, target, _ctx in loader:\n            scale_lr = min(1.0, (step + 1) / WARMUP)\n            for group in optimizer.param_groups:\n                group[\"lr\"] = group[\"base_lr\"] * scale_lr\n            days = days.cuda(non_blocking=True)\n            mask = mask.cuda(non_blocking=True)\n            target = target.cuda(non_blocking=True)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=\"cuda\", dtype=torch.float16):\n                pred, previous, _correction = model(days, mask)\n            loss = F.l1_loss(pred.float(), target.float())\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            scaler.step(optimizer)\n            scaler.update()\n            total_abs += (pred.detach() - target.float()).abs().sum().item()\n            base_abs += (previous.detach() - target.float()).abs().sum().item()\n            total_y += target.float().sum().item()\n            step += 1\n            if rank == 0 and (step == 1 or step % 50 == 0):\n                mix = torch.sigmoid(model.module.mix).item()\n                print(\n                    f\"step {step} epoch {epoch} loss {loss.item():.1f} mix {mix:.3f} \"\n                    f\"gpu_mb {round(torch.cuda.memory_allocated() / 2**20)}\",\n                    flush=True,\n                )\n        if rank == 0:\n            print(\n                f\"epoch {epoch} train WAPE {total_abs / max(total_y, 1):.4f} \"\n                f\"copy-week WAPE {base_abs / max(total_y, 1):.4f}\",\n                flush=True,\n            )\n\n\n@torch.no_grad()\ndef score_next_weeks(model, days, series, start, end):\n    model.eval()\n    model_abs = base_abs = denom = 0.0\n    cursor = start\n    while cursor <= end:\n        horizon_days = min(HORIZON, (end - cursor).days + 1)\n        target = days.index(cursor)\n        batch_days = []\n        batch_mask = []\n        actual = []\n        previous = []\n        for route_i in range(len(ROUTES)):\n            ctx = series[route_i][max(0, target - MAX_DAYS):target]\n            padded = np.zeros((MAX_DAYS, 24), np.float32)\n            mask = np.zeros(MAX_DAYS, np.float32)\n            padded[-len(ctx):] = ctx\n            mask[-len(ctx):] = 1.0\n            batch_days.append(padded)\n            batch_mask.append(mask)\n            actual.append(series[route_i][target:target + horizon_days].reshape(-1))\n            previous.append(series[route_i][target - HORIZON:target].reshape(-1)[: horizon_days * 24])\n        pred, _prev, _correction = model(\n            torch.tensor(np.stack(batch_days)).cuda(),\n            torch.tensor(np.stack(batch_mask)).cuda(),\n        )\n        pred = pred[:, : horizon_days * 24].float().cpu().numpy()\n        y = np.stack(actual)\n        base = np.stack(previous)\n        model_abs += np.abs(pred - y).sum()\n        base_abs += np.abs(base - y).sum()\n        denom += y.sum()\n        print(\n            f\"week {cursor.isoformat()} days {horizon_days} \"\n            f\"wape {np.abs(pred - y).sum() / max(y.sum(), 1):.4f} \"\n            f\"copy {np.abs(base - y).sum() / max(y.sum(), 1):.4f}\",\n            flush=True,\n        )\n        cursor += timedelta(days=7)\n    return model_abs, base_abs, denom\n\n\n@torch.no_grad()\ndef forecast_submission(model, days, series):\n    model.eval()\n    future_days = list(daterange(FORECAST_START, FORECAST_END))\n    known = series.shape[1]\n    extended = np.zeros((len(ROUTES), known + len(future_days), 24), np.float32)\n    extended[:, :known] = series\n    calendar = days + future_days\n    predictions = {}\n    cursor = FORECAST_START\n    while cursor <= FORECAST_END:\n        horizon_days = min(HORIZON, (FORECAST_END - cursor).days + 1)\n        target = calendar.index(cursor)\n        batch_days = []\n        batch_mask = []\n        for route_i in range(len(ROUTES)):\n            ctx = extended[route_i][max(0, target - MAX_DAYS):target]\n            padded = np.zeros((MAX_DAYS, 24), np.float32)\n            mask = np.zeros(MAX_DAYS, np.float32)\n            padded[-len(ctx):] = ctx\n            mask[-len(ctx):] = 1.0\n            batch_days.append(padded)\n            batch_mask.append(mask)\n        pred, _prev, _correction = model(\n            torch.tensor(np.stack(batch_days)).cuda(),\n            torch.tensor(np.stack(batch_mask)).cuda(),\n        )\n        pred = pred.float().cpu().numpy()\n        for route_i, route in enumerate(ROUTES):\n            flat = pred[route_i]\n            for offset in range(horizon_days):\n                day = cursor + timedelta(days=offset)\n                hours = flat[offset * 24:(offset + 1) * 24]\n                extended[route_i, target + offset] = hours\n                for hour, value in enumerate(hours):\n                    predictions[(route, day.isoformat(), hour)] = max(0.0, float(value))\n        cursor += timedelta(days=7)\n    return predictions\n\n\ndef write_submission(predictions):\n    import pandas as pd\n\n    submission = pd.read_csv(next(Path(\"/kaggle/input\").rglob(\"test_submission.csv\")), sep=\";\", encoding=\"utf-8-sig\")\n    filled = []\n    missing = 0\n    for row in submission.itertuples(index=False):\n        key = (int(row.route), str(row.date)[:10], int(row.hour))\n        if key not in predictions:\n            missing += 1\n            filled.append(0)\n        else:\n            filled.append(int(round(predictions[key])))\n    submission[\"prediction\"] = filled\n    submission.to_csv(OUT / \"submission.csv\", sep=\";\", index=False)\n    print(\n        \"submission\",\n        len(submission),\n        \"missing\",\n        missing,\n        \"mean\",\n        round(sum(filled) / len(filled), 1),\n        \"zeros\",\n        sum(value == 0 for value in filled),\n        flush=True,\n    )\n\n\ndef build_optimizer(model):\n    encoder, head = [], []\n    for name, param in model.named_parameters():\n        if not param.requires_grad:\n            continue\n        if \"day_proj\" in name or name.endswith(\"head.weight\") or name.endswith(\"head.bias\") or name.endswith(\"mix\"):\n            head.append(param)\n        else:\n            encoder.append(param)\n    optimizer = torch.optim.AdamW(\n        [\n            {\"params\": encoder, \"lr\": ENC_LR, \"weight_decay\": 0.01},\n            {\"params\": head, \"lr\": HEAD_LR, \"weight_decay\": 0.0},\n        ]\n    )\n    optimizer.param_groups[0][\"base_lr\"] = ENC_LR\n    optimizer.param_groups[1][\"base_lr\"] = HEAD_LR\n    return optimizer\n\n\ndef main():\n    rank = int(os.environ[\"LOCAL_RANK\"])\n    world = int(os.environ[\"WORLD_SIZE\"])\n    torch.manual_seed(42)\n    days, series = load_series()\n    train_targets = target_starts(days, TRAIN_END)\n    all_targets = target_starts(days, TEST_END)\n    dist.init_process_group(backend=\"nccl\", timeout=timedelta(hours=3))\n    torch.cuda.set_device(rank)\n    torch.backends.cuda.matmul.allow_tf32 = True\n    if rank == 0:\n        sep_i = days.index(TEST_START)\n        print(\n            f\"days {days[0].isoformat()}..{days[-1].isoformat()} \"\n            f\"train weeks {len(train_targets)} x {len(ROUTES)} routes \"\n            f\"context on Sep 1 = {sep_i} days (cap {MAX_DAYS})\",\n            flush=True,\n        )\n        print(f\"ddp world {world} device {torch.cuda.get_device_name(rank)}\", flush=True)\n    model = WeekBert()\n    for param in model.bert.embeddings.parameters():\n        param.requires_grad = False\n    pin_alibi(model.bert.encoder)\n    model = model.cuda(rank)\n    model = DDP(model, device_ids=[rank], output_device=rank)\n    optimizer = build_optimizer(model)\n    try:\n        scaler = torch.amp.GradScaler(\"cuda\")\n    except TypeError:\n        scaler = torch.cuda.amp.GradScaler()\n\n    if rank == 0:\n        print(\"phase 1: next week from the raw hourly year, targets through Aug 31\", flush=True)\n    loader, sampler = make_loader(WeekSet(days, series, train_targets), rank, world)\n    fit(model, loader, sampler, optimizer, scaler, EPOCHS_TRAIN, rank)\n    del loader, sampler\n    dist.barrier()\n\n    if rank == 0:\n        print(\"score Sep-Oct as successive next weeks, true history only, before autumn fit\", flush=True)\n        model_abs, base_abs, denom = score_next_weeks(model.module, days, series, TEST_START, TEST_END)\n        wape = model_abs / max(denom, 1.0)\n        score = max(0.0, 1.0 - wape)\n        base_wape = base_abs / max(denom, 1.0)\n        awarded = points_of(score)\n        text = (\n            f\"next_week_wape={wape:.6f}\\n\"\n            f\"next_week_score={score:.6f}\\n\"\n            f\"next_week_points={awarded}\\n\"\n            f\"next_week_weighted={awarded * 2}\\n\"\n            f\"copy_week_wape={base_wape:.6f}\\n\"\n            f\"copy_week_score={max(0.0, 1.0 - base_wape):.6f}\\n\"\n        )\n        (OUT / \"metrics.txt\").write_text(text, encoding=\"utf-8\")\n        print(text, flush=True)\n    dist.barrier()\n\n    if rank == 0:\n        print(\"phase 2: one epoch including Sep-Oct next-week targets\", flush=True)\n    loader, sampler = make_loader(WeekSet(days, series, all_targets), rank, world)\n    fit(model, loader, sampler, optimizer, scaler, EPOCHS_ALL, rank)\n    dist.barrier()\n    if rank == 0:\n        torch.save(model.module.state_dict(), OUT / \"boarding_bert.pt\")\n        write_submission(forecast_submission(model.module, days, series))\n        print((OUT / \"metrics.txt\").read_text(encoding=\"utf-8\"), flush=True)\n    dist.barrier()\n    dist.destroy_process_group()\n\n\nif __name__ == \"__main__\":\n    main()\n", encoding="utf-8")
env = os.environ.copy()
env["NCCL_P2P_DISABLE"] = "1"
env["NCCL_IB_DISABLE"] = "1"
env["TOKENIZERS_PARALLELISM"] = "false"
subprocess.check_call([
    sys.executable, "-m", "torch.distributed.run",
    "--nproc_per_node=2", "--standalone",
    "/kaggle/working/train_ddp.py",
], env=env)
print(Path("/kaggle/working/metrics.txt").read_text(encoding="utf-8"))
